# Bronze Layer — Raw Ingestion from S3
**What this does:** Reads raw JSON files from S3 and loads them into Delta Lake Bronze tables. No cleaning — data stored exactly as received from the APIs.

In [0]:
import boto3
import json
import pandas as pd

# Fetch AWS credentials from Databricks secret scope (our locked vault)
# dbutils.secrets.get() = retrieves the value without ever showing it — prints as [REDACTED]
aws_access_key = dbutils.secrets.get(scope="crypto-pipeline-scope", key="AWS_ACCESS_KEY_ID")
aws_secret_key = dbutils.secrets.get(scope="crypto-pipeline-scope", key="AWS_SECRET_ACCESS_KEY")
s3_bucket      = dbutils.secrets.get(scope="crypto-pipeline-scope", key="S3_BUCKET")

# boto3.client() = opens a secure, authenticated connection to AWS S3
s3 = boto3.client(
    "s3",
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    region_name="ap-southeast-1"
)

# Quick connection test — list a few files to confirm it works
response = s3.list_objects_v2(Bucket=s3_bucket, Prefix="prices/", MaxKeys=5)
print(f"S3 connected! Files found under prices/: {response.get('KeyCount', 0)}")
for obj in response.get("Contents", []):
    print(" -", obj["Key"])

In [0]:
# ── HELPER 1: read every .json file from a given S3 folder ──────────────────
# We write this once so it works for all 3 data sources (prices, fear_greed, news)
def read_json_files_from_s3(prefix):
    all_records = []
    # paginator handles S3's 1000-file limit — automatically fetches all pages
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=s3_bucket, Prefix=prefix)
    for page in pages:
        for obj in page.get("Contents", []):
            if obj["Key"].endswith(".json"):
                file_obj = s3.get_object(Bucket=s3_bucket, Key=obj["Key"])
                raw_text = file_obj["Body"].read().decode("utf-8")
                data = json.loads(raw_text)
                if isinstance(data, list):
                    all_records.extend(data)
                else:
                    all_records.append(data)
    return all_records


# ── HELPER 2: convert a list of dicts into a Spark DataFrame ────────────────
# prices files use "records" key, sentiment files use "data" key — we handle both
# Any nested dict/list gets stringified so Spark can store it as a plain text column
def to_spark_df(records):
    flat = []
    for r in records:
        if isinstance(r, dict) and "records" in r:
            # prices: extract the inner array of coin objects
            flat.extend(r["records"])
        elif isinstance(r, dict) and "data" in r:
            # sentiment: the "data" field can be a list or a single object
            data = r["data"]
            if isinstance(data, list):
                flat.extend(data)
            else:
                flat.append(r)  # keep the whole row, "data" will be stringified below
        else:
            flat.append(r)

    # Stringify any remaining nested types — Spark columns must be simple values
    cleaned = []
    for row in flat:
        cleaned.append({
            k: (json.dumps(v) if isinstance(v, (dict, list)) else v)
            for k, v in row.items()
        })

    # pandas DataFrame = in-memory table; spark.createDataFrame() works fine with it
    # This avoids sparkContext.parallelize() which is blocked in Serverless
    return spark.createDataFrame(pd.DataFrame(cleaned))


# ── READ PRICES ──────────────────────────────────────────────────────────────
prices_records = read_json_files_from_s3("prices/")
print(f"Prices records loaded: {len(prices_records)}")

df_prices_raw = to_spark_df(prices_records)
print(f"Columns: {df_prices_raw.columns}")
display(df_prices_raw)

In [0]:
# ── READ SENTIMENT ───────────────────────────────────────────────────────────
# fear_greed/ and news/ are top-level folders in S3 — not inside a sentiment/ folder
# This matches the s3_prefix used in fetch_sentiment.py lines 203-204
fear_greed_records = read_json_files_from_s3("fear_greed/")
news_records       = read_json_files_from_s3("news/")

print(f"Fear & Greed records loaded: {len(fear_greed_records)}")
print(f"News records loaded: {len(news_records)}")

df_fear_greed_raw = to_spark_df(fear_greed_records)
df_news_raw       = to_spark_df(news_records)

print(f"Fear & Greed columns: {df_fear_greed_raw.columns}")
print(f"News columns: {df_news_raw.columns}")
display(df_fear_greed_raw)

In [0]:
# ── WRITE TO DELTA LAKE BRONZE TABLES ───────────────────────────────────────
# format("delta")      = use Delta Lake format (gives us time travel + schema enforcement)
# mode("overwrite")    = replace fully on each run — safe to re-run, no duplicates
# saveAsTable()        = registers as a named SQL-queryable table in Databricks
df_prices_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_prices")
df_fear_greed_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_fear_greed")
df_news_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_news")

print("Bronze tables written:")
print("  bronze_prices")
print("  bronze_fear_greed")
print("  bronze_news")

In [0]:
# ── VERIFY ───────────────────────────────────────────────────────────────────
# SHOW TABLES lists every table in the current database
# You should see all 3 bronze tables below
display(spark.sql("SHOW TABLES"))